# Route Agent V1 —— 路线 → Claude 生成配乐歌单

不接 Spotify 的最小可行版本。核心逻辑：

1. 用 Google Maps 拿一条真实路线的事实数据（距离、时长、途经街道）
2. 把路线信息 + 你的口味档案 + 当下时间交给 Claude
3. Claude 判断这条路的「性格」，写一段叙事，配一份带能量曲线的歌单
4. 打印结果，同时存一份 Markdown 存档

**用法**：从上到下依次运行每个 cell。最后一个 cell 是可以反复修改起点/终点、反复重跑的部分，
前面的 cell（安装依赖、定义函数）通常只需要跑一次。

## 0. 准备工作（只需要做一次）

- **Anthropic API Key**：https://console.anthropic.com/ → API Keys → Create Key
- **Google Maps API Key**：https://console.cloud.google.com/ → 新建项目 → 启用 "Directions API" → 创建凭据

拿到两把 key 后，在这个 notebook 同一个文件夹下新建一个 `.env` 文件，写入：

```
ANTHROPIC_API_KEY=sk-ant-xxxxxx
GOOGLE_MAPS_API_KEY=AIzaxxxxxx
```

`.env` 不要分享给任何人、不要传到公开的地方。

In [ ]:
# 安装依赖（第一次运行时取消下面这行的注释）
# %pip install anthropic requests python-dotenv nbformat --quiet

In [ ]:
import os
import re
import json
import datetime
import requests
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

GOOGLE_MAPS_API_KEY = os.getenv("GOOGLE_MAPS_API_KEY")
ANTHROPIC_API_KEY = os.getenv("ANTHROPIC_API_KEY")
CLAUDE_MODEL = "claude-sonnet-5"

assert GOOGLE_MAPS_API_KEY, "没有找到 GOOGLE_MAPS_API_KEY，检查 .env 文件"
assert ANTHROPIC_API_KEY, "没有找到 ANTHROPIC_API_KEY，检查 .env 文件"
print("Key 已加载 ✓")

## 1. Route Geometry —— 拿路线的原始事实

只负责拿数据，不做任何"理解"，理解的部分留给 Claude。

In [ ]:
def get_route(origin: str, destination: str, mode: str = "walking") -> dict:
    """
    调用 Google Maps Directions API，返回这条路线的基础事实：
    距离、时长、途经的街道/地标名称
    """
    url = "https://maps.googleapis.com/maps/api/directions/json"
    params = {
        "origin": origin,
        "destination": destination,
        "mode": mode,
        "language": "ja",  # 街道名用日语更准确，Claude 读得懂
        "key": GOOGLE_MAPS_API_KEY,
    }
    resp = requests.get(url, params=params, timeout=10)
    resp.raise_for_status()
    data = resp.json()

    if data.get("status") != "OK":
        raise RuntimeError(f"路线请求失败：{data.get('status')} - {data.get('error_message', '')}")

    leg = data["routes"][0]["legs"][0]

    steps = []
    for step in leg["steps"]:
        clean = re.sub("<[^<]+?>", " ", step["html_instructions"]).strip()
        steps.append({
            "instruction": clean,
            "distance": step["distance"]["text"],
            "duration": step["duration"]["text"],
        })

    return {
        "origin": leg["start_address"],
        "destination": leg["end_address"],
        "distance": leg["distance"]["text"],
        "duration": leg["duration"]["text"],
        "mode": mode,
        "steps": steps,
    }

## 2. 口味档案

从同目录下的 `taste.md` 读取。第一次用之前，记得先花几分钟把 `taste.md` 填一下——
这份文件的详细程度，直接决定歌单准不准。

In [ ]:
def load_taste(path: str = "taste.md") -> str:
    if not os.path.exists(path):
        return "（用户还没有填写 taste.md，先用通用口味判断）"
    with open(path, "r", encoding="utf-8") as f:
        return f.read()

## 3. 路线理解 + 配乐 —— 交给 Claude

Claude 一次性完成三件事：判断路线性格（Route Character）、写路线叙事（Route Narrative）、
设计带能量曲线的歌单。刻意要求它不要编造假的精确数字，只做定性判断。

In [ ]:
SYSTEM_PROMPT = """你是一个「移动音乐体验设计师」。你的任务不是根据交通方式推荐音乐，
而是理解用户正在经历的这一段具体路线，为这条路线本身配一份有起承转合的原声带。

你需要完成三层理解，再给出结果：

1. Route Character（路线性格）：根据途经的地名、街道描述、区域气质，判断这条路从"什么感觉"
   走向"什么感觉"（例如：从高密度商业区 走向 安静住宅区；或者：始终是日常生活感，没有明显转折）。
   不要瞎编具体数据（不要编"商业指数 8.5"这种假精确的数字），用定性描述就好。

2. Route Narrative（路线叙事）：把上面的判断，写成 2-3 句话的"这段旅程是什么感觉"的描述。

3. Soundtrack（配乐）：设计一份歌单，歌曲数量根据总时长估算（大约每首歌 3-4 分钟）。
   歌单要有能量曲线，呼应路线性格的变化（比如从热闹到平静，音乐也应该有对应的起伏）。
   优先从用户的口味档案里选歌/选相似风格的歌；如果口味档案信息不够，可以合理推荐符合
   这条路线氛围、且和用户已知口味相近的歌曲，但要在 reason 里说明为什么适合。

只输出一个 JSON 对象，不要有任何 JSON 之外的文字、不要用 markdown 代码块包裹。JSON 结构如下：

{
  "route_character": {
    "start_feel": "起点区域给人的感觉",
    "end_feel": "终点区域给人的感觉",
    "arc_type": "用一个短语概括这条路的性格转变类型"
  },
  "narrative": "2-3句话的路线叙事",
  "playlist": [
    {
      "order": 1,
      "title": "歌名",
      "artist": "艺人",
      "segment": "start / middle / end 三选一，表示这首歌对应路线的哪一段",
      "energy": 1到10的整数,
      "reason": "为什么这首歌适合放在这个位置，一句话"
    }
  ]
}
"""

def generate_soundtrack(route_info: dict, taste: str, time_context: str) -> dict:
    client = Anthropic(api_key=ANTHROPIC_API_KEY)

    steps_text = "\n".join(
        f"- {s['instruction']}（{s['distance']}, {s['duration']}）" for s in route_info["steps"]
    )

    user_prompt = f"""这是我接下来要走的一段路，请为它配乐。

【路线事实】
起点：{route_info['origin']}
终点：{route_info['destination']}
出行方式：{route_info['mode']}
总距离：{route_info['distance']}
总时长：{route_info['duration']}

途经：
{steps_text}

【当下时间】
{time_context}

【我的口味档案】
{taste}

请按系统提示的格式输出 JSON。"""

    response = client.messages.create(
        model=CLAUDE_MODEL,
        max_tokens=3000,
        system=SYSTEM_PROMPT,
        messages=[{"role": "user", "content": user_prompt}],
    )

    raw_text = response.content[0].text.strip()
    if raw_text.startswith("```"):
        raw_text = raw_text.strip("`")
        if raw_text.startswith("json"):
            raw_text = raw_text[4:]

    return json.loads(raw_text)

## 4. 展示结果

In [ ]:
def render_result(route_info: dict, result: dict) -> str:
    lines = []
    lines.append(f"# {route_info['origin']} → {route_info['destination']}")
    lines.append(f"\n{route_info['distance']} · {route_info['duration']} · {route_info['mode']}\n")

    rc = result["route_character"]
    lines.append("## 路线性格")
    lines.append(f"起点感觉：{rc['start_feel']}")
    lines.append(f"终点感觉：{rc['end_feel']}")
    lines.append(f"转变类型：{rc['arc_type']}\n")

    lines.append("## 路线叙事")
    lines.append(result["narrative"] + "\n")

    lines.append("## 原声带")
    for song in result["playlist"]:
        bar = "█" * song["energy"] + "░" * (10 - song["energy"])
        lines.append(
            f"{song['order']:>2}. [{song['segment']:>6}] {bar} "
            f"{song['title']} — {song['artist']}"
        )
        lines.append(f"       └ {song['reason']}")

    return "\n".join(lines)

## 5. 运行

改下面这个 cell 里的起点/终点/出行方式，然后运行这一个 cell 加下一个 cell。
想试新路线时，只需要改这里、重新运行，不用动前面的函数定义。

In [ ]:
origin = "渋谷駅"
destination = "代官山駅"
mode = "walking"  # walking / transit / bicycling

In [ ]:
route_info = get_route(origin, destination, mode)
taste = load_taste()

now = datetime.datetime.now()
weekday_cn = ["周一", "周二", "周三", "周四", "周五", "周六", "周日"][now.weekday()]
time_context = now.strftime(f"%Y-%m-%d {weekday_cn} %H:%M")

result = generate_soundtrack(route_info, taste, time_context)
output = render_result(route_info, result)
print(output)

# 存一份记录，方便回看，也是以后做"反馈闭环"的数据基础
os.makedirs("history", exist_ok=True)
filename = f"history/{now.strftime('%Y%m%d_%H%M')}_{origin}_{destination}.md".replace(" ", "")
with open(filename, "w", encoding="utf-8") as f:
    f.write(output)
print(f"\n（已保存到 {filename}）")

---
### 遇到报错怎么办

- `AssertionError`（没找到 key）：检查 `.env` 文件名和内容
- `路线请求失败：REQUEST_DENIED`：Google Cloud 那边没启用 Directions API
- `路线请求失败：ZERO_RESULTS`：地名不够明确，换成带"站/駅"的写法试试
- Claude 返回内容解析失败（`json.decoder.JSONDecodeError`）：重新跑一次通常就好，
  多是模型输出里混了多余文字；如果反复出现，把报错信息发给我看